# Hello Kaggle Community! 👋
# 🐈‍⬛ Black Cat SWE Agent | Compact Challenger

**Objective.** Beat our **0.06 public resolution-rate anchor** with a runnable agent that stays inside the competition compute budget. The previous v3 scoring job returned *Notebook Exceeded Allowed Compute* and no score. That error does not identify its cause, so this version caps each issue's work instead of claiming a diagnosis.

| Evidence | Decision |
|---|---|
| Public issue reports can be brief; 129 public tasks cover four repositories | Extract exact identifiers, then search source and nearby tests |
| The [community 0.12 notebook](https://www.kaggle.com/code/romanrozen/gemma-eda-baseline-for-a-start-lb-top-1) demonstrates useful localization and patch checks | Use grep-first navigation, reproduction, compilation, and scratch-file cleanup |
| v3 used a 16,384 output ceiling and 4,096 thinking budget, then failed compute | Restore the scored anchor's smaller generation settings and add a four-minute per-issue cap |

**Challenge design:** one Gemma 4 agent; bounded search → reproduce → focused edit → targeted check → explicit `submit_patch()`. The original scored archive remains available as `anchor_submission.zip`. The challenger is unscored until Kaggle completes this submission.

Sources: [competition rules and scoring](https://www.kaggle.com/competitions/gemma-4-developer-agent/overview), [community notebook](https://www.kaggle.com/code/romanrozen/gemma-eda-baseline-for-a-start-lb-top-1).

In [ ]:
from pathlib import Path
from collections import Counter
import hashlib, json, platform, re, zipfile
import yaml

WORKING = Path('/kaggle/working') if platform.system() == 'Linux' and Path('/kaggle/working').exists() else Path.cwd()
WORKING.mkdir(parents=True, exist_ok=True)
print(r"""
 /\_/\\
( o.o )  BLACK CAT // COMPACT CHALLENGER
 > ^ <   grep -> reproduce -> patch -> check -> submit
         Chief Whisker Officer: "Scratch files sleep in /tmp."
""")
print(f'Python {platform.python_version()} | output: {WORKING}')

candidates = [Path.cwd() / 'tasks.jsonl', Path('/kaggle/input/gemma-4-developer-agent/tasks.jsonl')]
input_root = Path('/kaggle/input')
if input_root.exists():
    candidates.extend(input_root.rglob('tasks.jsonl'))
task_path = next((p for p in candidates if p.is_file()), None)
if task_path:
    with task_path.open(encoding='utf-8') as handle:
        tasks = [json.loads(line) for line in handle if line.strip()]
    counts = Counter(row['repo'] for row in tasks)
    one_file = sum(len(set(re.findall(r'^\+\+\+ b/(.+)$', row.get('patch',''), re.M))) == 1 for row in tasks)
    multi_hunk = sum(len(re.findall(r'^@@ ', row.get('patch',''), re.M)) > 1 for row in tasks)
    print(f'Public issues: {len(tasks)} | repositories: {len(counts)} | one-file fixes: {one_file}/{len(tasks)} | multi-hunk fixes: {multi_hunk}/{len(tasks)}')
    try:
        import matplotlib.pyplot as plt
        fig, ax = plt.subplots(figsize=(7, 2.5), facecolor='#0b1220')
        ax.set_facecolor('#111c2e')
        names, values = zip(*sorted(counts.items(), key=lambda item: item[1]))
        ax.barh(names, values, color='#39bdf8')
        ax.set_title('Public development issues by repository', color='white')
        ax.tick_params(colors='white')
        for spine in ax.spines.values(): spine.set_visible(False)
        fig.tight_layout()
        plt.show()
    except ImportError:
        print('Matplotlib unavailable; continuing without the optional chart.')
else:
    print('No public task index attached; archive generation remains available.')

## ⚙️ Agent and compute guard

The archive contains declarative YAML and a short prompt. `eval_config.yaml` limits each agent session to **4 minutes, 24 tool calls, 40 turns, and 60 seconds per command**. This is a resource safeguard, not a measured scoring gain. The benchmark's global limit is 12 hours, including sandbox setup. The model and nine registered tools match the working 0.06 anchor.

In [ ]:
PRIMARY = {'agent.yaml': 'name: black_cat_compact\nmodel: gemma-4-31b-it-qat-w4a16-ct\ndescription: Budget-aware software engineer for focused repository fixes.\ninstruction: !include prompts/system.md\ntools:\n  - run_command\n  - read_file\n  - edit_file\n  - write_file\n  - get_status\n  - submit_patch\n  - get_code_neighbors\n  - search_similar_code\n  - get_code_subgraph\ngenerate_content_config: !include configs/sampling.yaml\n', 'configs/sampling.yaml': 'temperature: 0.15\ntop_p: 0.90\nmax_output_tokens: 8192\nseed: 42\nthinking_config:\n  thinking_level: MEDIUM\n  thinking_budget: 2048\n  include_thoughts: false\n', 'prompts/system.md': 'You are Black Cat, an autonomous Python engineer in /workspace. Solve the issue with a real, minimal patch. A fresh checkout will run hidden tests. Finish by calling submit_patch(). Work within the stated task budget; make tool calls promptly and keep outputs short.\n\n1. Decode the issue: ignore PR-template boilerplate, keep the substantive request, exact identifiers, expected behavior, and edge cases. A short title still requires a best-effort implementation.\n2. Locate: use the initial file listing, then one bounded `rg -n` search for exact identifiers or error text (`grep -Rn` if rg is missing). Read only relevant ranges and nearby tests/examples. If the first route misses, search a second related symbol. Use graph tools only for known symbol names; if unavailable, return to source search. Do not treat natural-language text as a symbol query.\n3. Reproduce: when feasible, run a tiny behavior check with `python3 -c` or put a scratch script under /tmp. Confirm the failure before editing. Never leave scratch files in /workspace.\n4. Fix: change the earliest incorrect source layer with focused `edit_file` calls. Copy old text exactly and ensure it matches uniquely. Multiple coordinated edits are fine when behavior requires them. Preserve unrelated behavior and public APIs. Do not modify tests, hidden data, harness files, or broad formatting. Python example source may be changed when it is the implementation target.\n5. Verify: compile changed Python with `python3 -m py_compile`, rerun the reproduction, and run the closest targeted test if time allows. Inspect failures; do not claim a pass without tool output. Avoid full test suites, package installs, and network access.\n6. Review: check `git status --short`, `git diff --check`, and the relevant diff. Remove accidental files. Call get_status() around the middle of the budget. When little time remains, stop exploring and submit the best defensible patch.\n7. Call submit_patch() as the final tool action. It is free, captures intended new files, and ends the agent session. Then summarize only verified changes and checks.\n\nCommands have a 60-second cap in this submission and truncated output. If a tool fails twice, take a simpler route. Never fabricate source content, edits, or test results.\n', 'eval_config.yaml': 'evaluation:\n  timeout_seconds: 60\n  max_tool_calls: 24\n  max_time_minutes: 4\n  max_turns: 40\n'}
ANCHOR = {'agent.yaml': 'name: black_cat_swe_agent\nmodel: gemma-4-31b-it-qat-w4a16-ct\ndescription: Repository-agnostic software engineering agent for focused Python fixes.\ninstruction: !include prompts/system.md\ntools:\n  - run_command\n  - read_file\n  - edit_file\n  - write_file\n  - get_status\n  - submit_patch\n  - get_code_neighbors\n  - search_similar_code\n  - get_code_subgraph\ngenerate_content_config: !include configs/sampling.yaml\n', 'configs/sampling.yaml': 'temperature: 0.15\ntop_p: 0.90\nmax_output_tokens: 8192\nseed: 42\nthinking_config:\n  thinking_level: MEDIUM\n  thinking_budget: 2048\n  include_thoughts: false\n', 'prompts/system.md': 'You are Black Cat, an autonomous software engineer working on one Python repository issue. Your deliverable is a minimal, working source-code patch under /workspace. The benchmark checks the patch in a fresh checkout with separate tests. Work independently and finish by calling submit_patch().\n\nOperating priorities: correctness, focused navigation, reproducible edits, and useful work within the time and tool budget. Never invent a test result or claim a file was changed before a tool confirms it.\n\n1. Orient quickly. Read the problem statement and hints. Identify exact error strings, public behavior, expected types, and likely symbols. The initial prompt includes a workspace listing. If it identifies an obvious file, read that file. Otherwise make one bounded discovery call using run_command with rg over relevant Python paths. Use search_similar_code only with a real symbol name, not a natural-language sentence; it resolves existing graph nodes, not arbitrary queries. Use graph neighbors when a caller or dependency relationship changes the fix. If graph tools are missing or return errors, use rg and read_file.\n2. Diagnose before editing. Inspect the implementation and the nearest relevant tests or examples. State a short hypothesis internally and distinguish observed behavior from guesses. Favor one small source edit over broad rewrites. Preserve signatures, existing conventions, compatibility, and error wording required by the issue. For feature requests, implement the specified behavior end to end, including edge cases that the issue names.\n3. Edit precisely with edit_file. If a matching block is ambiguous, read a narrower range before retrying. Use write_file only for a new implementation file or when a narrow replacement is impossible. Do not alter tests, snapshots, generated files, pytest.ini, conftest.py, package metadata, or unrelated formatting to make checks pass. In tasks involving executable documentation examples, docs_src or equivalent example source may be the implementation target.\n4. Verify economically. Prefer a targeted existing test or a tiny inline reproduction for the changed behavior. Specify the exact test module or test name; never run an entire repository test suite. Commands have a 300-second ceiling and truncated output. Do not install packages or access the network. If a test fails, diagnose whether it is related to the change. Fix a real regression if time permits. Ignore unrelated pre-existing environment failures after recording the uncertainty internally.\n5. Inspect git diff --check and git diff --stat, then inspect the relevant diff. Remove temporary files from /workspace; put scratch scripts in /tmp. Do not put expected-answer strings or issue-specific hardcoded patches into the repository. Never inspect hidden solution files or future commits. If time or tool calls run low, preserve the best working edit and finish promptly.\n6. Call submit_patch() as the final tool action. It captures git diff HEAD, including intended new files. Check the returned patch_size and files_changed. If no defensible edit is possible, submit the empty state honestly rather than inventing a change. Give a concise final summary with the actual verification status.\n\nTool budget guide: spend roughly 2-4 calls locating code, 2-5 calls implementing, and 1-3 calls validating. This is a guide, not a hard limit; complex issues may need more. Do not use an analysis sub-agent unless one is provided by the configuration. The harness may continue a turn if you stop without submit_patch(), so make that call explicitly.\n'}
FALLBACK = {'agent.yaml': 'name: black_cat_fallback\nmodel: gemma-4-31b-it-qat-w4a16-ct\ninstruction: !include prompts/system.md\ntools: [run_command, read_file, edit_file, get_status, submit_patch]\ngenerate_content_config:\n  temperature: 0.1\n  max_output_tokens: 4096\n', 'prompts/system.md': 'Solve the issue in /workspace. Inspect the relevant source, make a focused change, run a small check, inspect git diff, and call submit_patch(). Do not edit tests or harness files.\n', 'eval_config.yaml': 'evaluation:\n  timeout_seconds: 60\n  max_tool_calls: 24\n  max_time_minutes: 4\n  max_turns: 40\n'}


In [ ]:
class IncludeLoader(yaml.SafeLoader):
    pass

IncludeLoader.add_constructor('!include', lambda loader, node: loader.construct_scalar(node))
ALLOWED = {'.yaml', '.yml', '.md', '.txt', '.py', '.json', '.safetensors'}
REGISTERED_TOOLS = {'run_command', 'read_file', 'edit_file', 'write_file', 'get_status',
                    'submit_patch', 'get_code_neighbors', 'search_similar_code', 'get_code_subgraph'}

def package(files, output):
    assert 'agent.yaml' in files and len(files) <= 10000
    for name, content in files.items():
        path = Path(name)
        assert not path.is_absolute() and '..' not in path.parts and path.suffix in ALLOWED
        assert isinstance(content, str) and content.strip()
    for name, content in files.items():
        if name.endswith(('.yaml', '.yml')):
            yaml.load(content, Loader=IncludeLoader)
        for include in re.findall(r'!include\s+([^\s]+)', content):
            target = (Path(name).parent / include).as_posix()
            assert target in files, (name, include)
    agent = yaml.load(files['agent.yaml'], Loader=IncludeLoader)
    assert agent['model'] == 'gemma-4-31b-it-qat-w4a16-ct'
    assert 'submit_patch' in agent['tools'] and set(agent['tools']) <= REGISTERED_TOOLS
    if 'eval_config.yaml' in files:
        budget = yaml.safe_load(files['eval_config.yaml'])['evaluation']
        assert 0 < budget['max_time_minutes'] <= 4
        assert 0 < budget['max_tool_calls'] <= 24
        assert 0 < budget['max_turns'] <= 40
        assert 0 < budget['timeout_seconds'] <= 60
    with zipfile.ZipFile(output, 'w') as archive:
        for name, content in sorted(files.items()):
            info = zipfile.ZipInfo(name, date_time=(1980, 1, 1, 0, 0, 0))
            info.compress_type = zipfile.ZIP_DEFLATED
            info.create_system = 3
            info.external_attr = 0o644 << 16
            archive.writestr(info, content.encode('utf-8'))
    with zipfile.ZipFile(output) as archive:
        assert archive.testzip() is None
        assert sorted(archive.namelist()) == sorted(files)
        assert all(archive.read(name) == files[name].encode('utf-8') for name in files)
    return output

primary_path = package(PRIMARY, WORKING / 'submission.zip')
anchor_path = package(ANCHOR, WORKING / 'anchor_submission.zip')
fallback_path = package(FALLBACK, WORKING / 'fallback_submission.zip')
manifest = {'selected': primary_path.name, 'sha256': hashlib.sha256(primary_path.read_bytes()).hexdigest(),
            'anchor_public_score': 0.06, 'challenger_score': None, 'budget': yaml.safe_load(PRIMARY['eval_config.yaml'])['evaluation']}
(WORKING / 'experiment_card.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
print('✅ FINAL PACKAGE CHECK PASSED')
for path in (primary_path, anchor_path, fallback_path):
    print(f'📦 {path.name}: {path.stat().st_size:,} bytes')
print('Challenger SHA-256:', manifest['sha256'])
print('Leaderboard score: pending Kaggle evaluation')

## 🐈‍⬛ Handoff

`submission.zip` is the **new trial**. `anchor_submission.zip` preserves the scored 0.06 agent; `fallback_submission.zip` is the simplest valid bundle. The notebook checks paths, includes, YAML, tool names, budgets, ZIP integrity, and byte-for-byte archive contents. It cannot certify issue resolution without the competition evaluator. **Promote this challenger only if its completed score beats the anchor; retain the anchor otherwise.**